All Dependencies

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
from glob import glob

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Mount Drive (Execute only if using Colab)

In [ ]:
'''from google.colab import drive
drive.mount('/content/drive')'''

"from google.colab import drive\ndrive.mount('/content/drive')"

Load Dataset

In [ ]:
file_path = "/content/drive/MyDrive/Tasks/Classifier/dataset/dataset"

# Initialise parameters
IMG_SIZE = 150
BATCH_SIZE = 32
EPOCHS_FROZEN = 20
EPOCHS_FINETUNE = 20
NUM_CLASSES = 3
CLASS_NAMES = ["no", "sphere", "vort"]
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

def load_dataset(dataset_dir):

  X_train = []
  y_train = []
  X_test = []
  y_test = []

  for d in os.listdir(dataset_dir):
    if d == "train":
      train_dir = os.path.join(dataset_dir, d)
      idx = 0
      for class_name in CLASS_NAMES:
        class_dir = os.path.join(train_dir, class_name)
        for img in os.listdir(class_dir):
          if img.endswith(".npy"):
            img_path = os.path.join(class_dir, img)
            img_data = np.load(img_path)
            img_data = np.transpose(img_data, (1, 2, 0))
            X_train.append(img_data)
            y_train.append(idx)
        idx += 1
    if d == "val":
      test_dir = os.path.join(dataset_dir, d)
      idx = 0
      for class_name in CLASS_NAMES:
        class_dir = os.path.join(test_dir, class_name)
        for img in os.listdir(class_dir):
          if img.endswith(".npy"):
            img_path = os.path.join(class_dir, img)
            img_data = np.load(img_path)
            img_data = np.transpose(img_data, (1, 2, 0))
            X_test.append(img_data)
            y_test.append(idx)
        idx += 1

  X_train = np.array(X_train)
  y_train = np.array(y_train)
  X_test = np.array(X_test)
  y_test = np.array(y_test)
  return X_train, y_train, X_test, y_test


# Convert to RGB img
def to_3ch(a):
    if a.shape[-1] == 1:
        a = np.repeat(a, 3, axis=-1)  # (N,H,W,1) → (N,H,W,3)
        print(f"  Converted 1ch → 3ch: {a.shape}")
    return a.astype(np.float32)

X_train, y_train, X_test, y_test = load_dataset(file_path)
X_train = to_3ch(X_train)
X_test = to_3ch(X_test)

Build Model

In [ ]:
def build_model():
    base = ResNet50(include_top=False, weights="imagenet",
                    input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.RandomFlip("horizontal_and_vertical")(inputs)
    x = layers.RandomRotation(0.5)(x)
    x = layers.RandomZoom((-0.1, 0.1))(x)
    x = keras.applications.resnet50.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="DeepLense_Classifier"), base

model, base_model = build_model()
model.summary()

model.compile(optimizer=optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

Train Model

In [ ]:
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))

history1 = model.fit(X_train, y_train, validation_data=(X_test, y_test),
    epochs=EPOCHS_FROZEN, batch_size=BATCH_SIZE, class_weight=class_weights,
    callbacks=[callbacks.EarlyStopping(patience=7, restore_best_weights=True),
               callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)],
    verbose=1)

# Fine-Tune Top ResNet Layers

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
print(f"Fine-tuning {sum(1 for l in base_model.layers if l.trainable)}/{len(base_model.layers)} layers")

model.compile(optimizer=optimizers.Adam(5e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history2 = model.fit(X_train, y_train, validation_data=(X_test, y_test),
    epochs=EPOCHS_FINETUNE, batch_size=BATCH_SIZE, class_weight=class_weights,
    callbacks=[callbacks.EarlyStopping(patience=7, restore_best_weights=True),
               callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)],
    verbose=1)

Plot Model

In [ ]:
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
p1 = len(history1.history['accuracy'])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))

a1.plot(acc, 'b', label='Train')
a1.plot(val_acc, 'r', label='Val')
a1.axvline(p1, color='gray', ls='--', label='Fine-tune start')
a1.set_title('Accuracy')
a1.legend()
a1.grid(alpha=0.3)

a2.plot(loss, 'b', label='Train')
a2.plot(val_loss, 'r', label='Val')
a2.axvline(p1, color='gray', ls='--', label='Fine-tune start')
a2.set_title('Loss')
a2.legend()
a2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Test Model

In [ ]:
y_proba = model.predict(X_test)
y_pred = np.argmax(y_proba, axis=1)

print("CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

y_bin = label_binarize(y_test, classes=[0, 1, 2])
fpr, tpr, roc_auc = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_proba.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(9, 7))
for i, c in enumerate(['#1f77b4', '#ff7f0e', '#2ca02c']):
    plt.plot(fpr[i], tpr[i], color=c, lw=2,
             label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.4f})')
plt.plot(fpr["micro"], tpr["micro"], 'r--', lw=2,
         label=f'Micro-avg (AUC = {roc_auc["micro"]:.4f})')
plt.plot([0,1],[0,1],'k--',alpha=0.3)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Lensing Classification (ResNet50 + ImageNet)')
plt.legend(fontsize=11); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("roc_curves.png", dpi=150); plt.show()

print("\nAUC SCORES:")
for i in range(NUM_CLASSES):
    print(f"  {CLASS_NAMES[i]:25s}: {roc_auc[i]:.4f}")
print(f"  {'Micro-average':25s}: {roc_auc['micro']:.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues'); plt.colorbar()
for i in range(3):
    for j in range(3):
        plt.text(j, i, str(cm[i,j]), ha='center', va='center',
                 color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=15)
plt.xticks([0,1,2], CLASS_NAMES, rotation=45); plt.yticks([0,1,2], CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

model.save("lens_classifier_resnet50.keras")
print("Model saved: lens_classifier_resnet50.keras")

Download Model (Execute only if using Colab)

In [ ]:
from google.colab import files
files.download("lens_classifier_resnet50.keras")